# Week 3 - Day 6,7 work

In [2]:
# Day 6 — Model Integration & Prediction

In [3]:
# Day 6 - Model Integration & Prediction

import pandas as pd
import numpy as np
import joblib

In [4]:
# Check available project files

import os

files = os.listdir()
print(files)

['.anaconda', '.bash_history', '.conda', '.condarc', '.continuum', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.ms-ad', '.python_history', '.redhat', '.vscode', '01_Dataset_Telco-Customer-Churn.csv', '3D Objects', 'anaconda3', 'anaconda_projects', 'AppData', 'Application Data', 'bar.png', 'company_data_csv.csv', 'Contacts', 'Cookies', 'data_xlsx.xlsx', 'Desktop', 'Documents', 'Downloads', 'employee_data_csv.csv', 'expenses_data_csv.csv', 'Favorites', 'foodfile.csv', 'IntelGraphicsProfiles', 'IPL_Data_ Analytic_projects', 'Links', 'Local Settings', 'matplotlib.ipynb', 'Microsoft', 'Music', 'My Documents', 'NetHood', 'New folder', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{53b39e88-18c4-11ea-a811-000d3aa4692b}.TM.blf', 'NTUSER.DAT{53b39e88-18c4-11ea-a811-000d3aa4692b}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{53b39e88-18c4-11ea-a811-000d3aa4692b}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'null_data.csv

In [5]:
# Load the feature-engineered customer dataset

df = pd.read_csv("telco_customer_churn_feature_engineered.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (7043, 25)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ...        Contract  \
0  No phone service             DSL             No  ...  Month-to-month   
1                No             DSL            Yes  ...        One year   
2                No             DSL            Yes  ...  Month-to-month   
3  No phone service             DSL            Yes  ...        One year   
4                No     Fiber optic             No  ...  Month-to-month   

  PaperlessBilling              PaymentMet

In [6]:
# Check dataset columns

print("Columns in dataset:")
for i, column in enumerate(df.columns, start=1):
    print(i, column)

Columns in dataset:
1 customerID
2 gender
3 SeniorCitizen
4 Partner
5 Dependents
6 tenure
7 PhoneService
8 MultipleLines
9 InternetService
10 OnlineSecurity
11 OnlineBackup
12 DeviceProtection
13 TechSupport
14 StreamingTV
15 StreamingMovies
16 Contract
17 PaperlessBilling
18 PaymentMethod
19 MonthlyCharges
20 TotalCharges
21 Churn
22 TotalChargesPerTenure
23 TenureGroup
24 MonthlyChargePerTenure
25 ChurnNumeric


In [7]:
# Prepare numeric features for batch prediction

numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "TotalChargesPerTenure",
    "MonthlyChargePerTenure"
]

X_batch = df[numeric_features]

print("Batch input shape:", X_batch.shape)
print(X_batch.head())

Batch input shape: (7043, 5)
   tenure  MonthlyCharges  TotalCharges  TotalChargesPerTenure  \
0       1           29.85         29.85              29.850000   
1      34           56.95       1889.50              55.573529   
2       2           53.85        108.15              54.075000   
3      45           42.30       1840.75              40.905556   
4       2           70.70        151.65              75.825000   

   MonthlyChargePerTenure  
0                  29.850  
1                   1.675  
2                  26.925  
3                   0.940  
4                  35.350  


In [8]:
# Find saved model files

import os

model_files = [
    f for f in os.listdir()
    if f.endswith((".pkl", ".joblib", ".pickle"))
]

print("Saved model files:")
print(model_files)

Saved model files:
[]


In [9]:
# Day 6 - Recreate the LTV prediction model

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

FEATURES = [
    "gender", "SeniorCitizen", "Partner", "Dependents", "tenure",
    "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
    "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
    "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
    "MonthlyCharges", "TotalCharges"
]

# Prepare data
model_df = df.copy()

model_df["TotalCharges"] = pd.to_numeric(
    model_df["TotalCharges"], errors="coerce"
)

model_df["TotalCharges"] = model_df["TotalCharges"].fillna(
    model_df["TotalCharges"].median()
)

# Use active customers for LTV calculation
active = model_df[model_df["Churn"] == "No"].copy()

# Calculate expected LTV
active["ExpectedRemainingMonths"] = (
    72 - active["tenure"]
).clip(lower=1)

active["Estimated_LTV"] = (
    active["MonthlyCharges"] *
    active["ExpectedRemainingMonths"]
)

X = active[FEATURES]
y = active["Estimated_LTV"]

# Separate feature types
numeric = X.select_dtypes(include=["number"]).columns.tolist()
categorical = X.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("numerical", "passthrough", numeric)
])

# Build and train model
ltv_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regression", LinearRegression())
])

ltv_model.fit(X, y)

print("LTV model trained successfully.")
print("Training records:", len(X))

LTV model trained successfully.
Training records: 5174


In [10]:
# Day 6 - Batch LTV Prediction

X_batch = df[FEATURES].copy()

batch_predictions = ltv_model.predict(X_batch)

df["Predicted_LTV"] = batch_predictions

print("Batch predictions completed.")
print("Total predictions:", len(df))
print(df[["customerID", "Predicted_LTV"]].head())

Batch predictions completed.
Total predictions: 7043
   customerID  Predicted_LTV
0  7590-VHVEG    2135.249124
1  5575-GNVDE    2206.804640
2  3668-QPYBK    3760.404297
3  7795-CFOCW    1207.229061
4  9237-HQITU    4926.026350


In [11]:
# Day 7 - FastAPI Prediction Endpoint

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(
    title="Customer LTV Prediction API",
    description="FastAPI service for single-customer LTV prediction",
    version="1.0.0"
)

class CustomerInput(BaseModel):
    tenure_months: float
    monthly_charges: float
    total_charges: float

@app.get("/")
def read_root():
    return {
        "message": "Customer LTV Prediction API",
        "status": "running"
    }

@app.post("/predict")
def predict_ltv(customer: CustomerInput):
    input_data = pd.DataFrame([{
        "tenure": customer.tenure_months,
        "MonthlyCharges": customer.monthly_charges,
        "TotalCharges": customer.total_charges,
        "TotalChargesPerTenure": (
            customer.total_charges / customer.tenure_months
            if customer.tenure_months > 0 else 0
        ),
        "MonthlyChargePerTenure": (
            customer.monthly_charges / customer.tenure_months
            if customer.tenure_months > 0 else 0
        )
    }])

    prediction = ltv_model.predict(input_data)[0]

    return {
        "predicted_ltv": float(prediction)
    }

print("FastAPI application created successfully.")

FastAPI application created successfully.


In [13]:
# Day 7 - Test Single Customer Prediction

# Take one complete customer record with all required features
test_customer = df[FEATURES].iloc[[0]].copy()

# Change numeric values for testing
test_customer["tenure"] = 12
test_customer["MonthlyCharges"] = 70.0
test_customer["TotalCharges"] = 840.0

# Update engineered features
test_customer["TotalChargesPerTenure"] = 840.0 / 12
test_customer["MonthlyChargePerTenure"] = 70.0 / 12

# Predict
prediction = ltv_model.predict(test_customer)[0]

print("Test customer prediction successful.")
print("Predicted LTV:", round(float(prediction), 2))

Test customer prediction successful.
Predicted LTV: 4185.84
